In [1]:
library(keras)
library(tensorflow)
library(tidyverse)
library(recipes)

Warning message:
"le package 'keras' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'tensorflow' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'tidyverse' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'ggplot2' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'tibble' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'tidyr' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'readr' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'dplyr' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'forcats' a été compilé avec la version R 4.2.3"
Warning message:
"le package 'lubridate' a été compilé avec la version R 4.2.3"
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.2     ✔ readr     2.1.4
✔ forcats   1.0.0     ✔ stringr   1.5.0
✔ ggplot2   3.4.2     ✔ tibble    3.2.1
✔ lubridate 1.9.2    

In [2]:
ConfusionMatrix <- function(y_pred, y_true) {
  Confusion_Mat <- table(y_true, y_pred)
  return(Confusion_Mat)
}
 
ConfusionDF <- function(y_pred, y_true) {
  Confusion_DF <- transform(as.data.frame(ConfusionMatrix(y_pred, y_true)),
                            y_true = as.character(y_true),
                            y_pred = as.character(y_pred),
                            Freq = as.integer(Freq))
  return(Confusion_DF)
}
 
Precision_micro <- function(y_true, y_pred, labels = NULL) {
  Confusion_DF <- ConfusionDF(y_pred, y_true)
 
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred))
  # this is not bulletproof since there might be labels missing (in strange cases)
  # in strange cases where they existed in training set but are missing from test ground truth and predictions.
 
  TP <- c()
  FP <- c()
  for (i in c(1:length(labels))) {
    positive <- labels[i]
   
    # it may happen that a label is never predicted (missing from y_pred) but exists in y_true
    # in this case ConfusionDF will not have these lines and thus the simplified code crashes
    # TP[i] <- as.integer(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"])
    # FP[i] <- as.integer(sum(Confusion_DF[which(Confusion_DF$y_true!=positive & Confusion_DF$y_pred==positive), "Freq"]))
   
    # workaround:
    # i don't want to change ConfusionDF since i don't know if the current behaviour is a feature or a bug.
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"]
    TP[i] <- if (length(tmp)==0) 0 else as.integer(tmp)
   
    tmp <- Confusion_DF[which(Confusion_DF$y_true!=positive & Confusion_DF$y_pred==positive), "Freq"]
    FP[i] <- if (length(tmp)==0) 0 else as.integer(sum(tmp))
  }
  Precision_micro <- sum(TP) / (sum(TP) + sum(FP))
  return(Precision_micro)
}
 
Recall_micro <- function(y_true, y_pred, labels = NULL) {
  Confusion_DF <- ConfusionDF(y_pred, y_true)
 
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred))
  # this is not bulletproof since there might be labels missing (in strange cases)
  # in strange cases where they existed in training set but are missing from test ground truth and predictions.
 
  TP <- c()
  FN <- c()
  for (i in c(1:length(labels))) {
    positive <- labels[i]
   
    # short version, comment out due to bug or feature of Confusion_DF
    # TP[i] <- as.integer(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"])
    # FP[i] <- as.integer(sum(Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred!=positive), "Freq"]))
   
    # workaround:
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred==positive), "Freq"]
    TP[i] <- if (length(tmp)==0) 0 else as.integer(tmp)
 
    tmp <- Confusion_DF[which(Confusion_DF$y_true==positive & Confusion_DF$y_pred!=positive), "Freq"]
    FN[i] <- if (length(tmp)==0) 0 else as.integer(sum(tmp))
  }
  Recall_micro <- sum(TP) / (sum(TP) + sum(FN))
  return(Recall_micro)
}
 
F1_Score_micro <- function(y_true, y_pred, labels = NULL) {
  if (is.null(labels) == TRUE) labels <- unique(c(y_true, y_pred)) # possible problems if labels are missing from y_*
  Precision <- Precision_micro(y_true, y_pred, labels)
  Recall <- Recall_micro(y_true, y_pred, labels)
  F1_Score_micro <- 2 * (Precision * Recall) / (Precision + Recall)
  return(F1_Score_micro)
}

In [3]:
dataNN<-read.csv("data_target_encoding_NN.csv",stringsAsFactors = T)
targets<-which(grepl('damage_grade',colnames(dataNN)))

In [4]:
# drop = F is used to preserve the structure of the data as data.frame (see https://www.r-bloggers.com/2018/02/r-tip-use-drop-false-with-data-frames/)

n<-ncol(dataNN)
correlation<-abs(cor(dataNN[,-targets,drop=F],dataNN[,targets,drop=F]))
selected<-c()
candidates<-1:(n-length(targets))

    #mRMR ranks the variables by taking account not only the correlation with the output, but also by avoiding redudant variables
    for (j in 1:n) {
        redundancy_score<-numeric(length(candidates))
        
        if (length(selected)>0) {
            # Compute the correlation between the selected variables and the candidates on the training set
            cor_selected_candidates<-cor(dataNN[,selected,drop=F],dataNN[,candidates,drop=F])
            # Compute the mean correlation for each candidate variable, across the selected variables
            redundancy_score<-apply(cor_selected_candidates,2,mean)
        }
        
        # mRMR: minimum Redundancy Maximum Relevancy
        mRMR_score<-correlation[candidates]-redundancy_score
        
        # Select the candidate variable that maximises the mRMR score
        selected_current<-candidates[which.max(mRMR_score)]
        selected<-c(selected,selected_current)
        
        # Remove the selected variables from the candidates
        candidates<-setdiff(candidates,selected_current)
    }
    
    rankingNN <- selected

In [5]:
rankingNN
colnames(dataNN)[rankingNN]

[1]  5 46 44 43 38 40 12 37 16  3  4 50 49 51  8  6  1 20  2 41 19 39 42  7 67
[26] 69  9 25 11 24 17 59 66 55 13 23 22 52 36 15 33 35 64 18 47 14 26 34 56 48
[51] 21 70 57 58 68 63 28 45 27 10 65 30 32 29 60 62 61 53 54 31

[1] "geo_level_3_mean_damage"               
 [2] "ground_floor_type_v"                   
 [3] "ground_floor_type_f"                   
 [4] "roof_type_x"                           
 [5] "foundation_type_r"                     
 [6] "foundation_type_w"                     
 [7] "has_superstructure_mud_mortar_stone"   
 [8] "foundation_type_i"                     
 [9] "has_superstructure_cement_mortar_brick"
[10] "geo_level_2_mean_damage"               
[11] "geo_level_2_sd_damage"                 
[12] "other_floor_type_q"                    
[13] "other_floor_type_j"                    
[14] "other_floor_type_s"                    
[15] "age"                                   
[16] "geo_level_3_sd_damage"                 
[17] "geo_level_1_mean_damage"               
[18] "has_superstructure_rc_engineered"      
[19] "geo_level_1_sd_damage"                 
[20] "roof_type_n"                           
[21] "has_superstructure_rc_non_engineered"  
[22] "foundation_type_u"                     
[23] "roof_type_q"                           
[24] "count_floors_pre_eq"                   
[25] "legal_ownership_status_a"              
[26] "legal_ownership_status_v"              
[27] "area_percentage"                       
[28] "has_secondary_use_rental"              
[29] "has_superstructure_adobe_mud"          
[30] "has_secondary_use_hotel"               
[31] "has_superstructure_timber"             
[32] "plan_configuration_d"                  
[33] "plan_configuration_u"                  
[34] "position_s"                            
[35] "has_superstructure_stone_flag"         
[36] "has_secondary_use_agriculture"         
[37] "count_families"                        
[38] "other_floor_type_x"                    
[39] "foundation_type_h"                     
[40] "has_superstructure_mud_mortar_brick"   
[41] "land_surface_condition_n"              
[42] "land_surface_condition_t"              
[43] "plan_configuration_q"                  
[44] "has_superstructure_bamboo"             
[45] "ground_floor_type_x"                   
[46] "has_superstructure_cement_mortar_stone"
[47] "has_secondary_use_institution"         
[48] "land_surface_condition_o"              
[49] "position_t"                            
[50] "ground_floor_type_z"                   
[51] "has_superstructure_other"              
[52] "legal_ownership_status_w"              
[53] "plan_configuration_a"                  
[54] "plan_configuration_c"                  
[55] "legal_ownership_status_r"              
[56] "plan_configuration_o"                  
[57] "has_secondary_use_industry"            
[58] "ground_floor_type_m"                   
[59] "has_secondary_use_school"              
[60] "height_percentage"                     
[61] "plan_configuration_s"                  
[62] "has_secondary_use_gov_office"          
[63] "has_secondary_use_other"               
[64] "has_secondary_use_health_post"         
[65] "plan_configuration_f"                  
[66] "plan_configuration_n"                  
[67] "plan_configuration_m"                  
[68] "position_j"                            
[69] "position_o"                            
[70] "has_secondary_use_use_police"

In [4]:
classConverter <- function(predict_data,test_data) {
    yhat<-data.frame(matrix(0,ncol = 1, nrow = nrow(predict_data)))
    y<-data.frame(matrix(0,ncol = 1, nrow = nrow(predict_data)))
    for (i in 1:nrow(predict_data)){
        yhat[i,]<-which.max(predict_data[i,])
        y[i,]<-which.max(test_data[i,]) 
    }
    mylist <- list(yhat,y)
}

In [5]:
# drop = F is used to preserve the structure of the data as data.frame (see https://www.r-bloggers.com/2018/02/r-tip-use-drop-false-with-data-frames/)
set.seed(2) 
CV_folds <- 5

targets<-which(grepl('damage_grade',colnames(dataNN)))
n<-ncol(dataNN[,-targets])
N<-nrow(dataNN[,-targets])
size_CV <-floor(N/CV_folds)

accuracy_vec <- data.frame(matrix(ncol = 3, nrow = 0))
colnames(accuracy_vec)<-c('n','fold','F1')
meanF1 <- data.frame(matrix(ncol = 2, nrow = 0))
colnames(meanF1)<-c('n','mean F1')

for (nb_features in seq(2,n,5)) {
    for (i in 1:CV_folds) {
    
        targets<-which(grepl('damage_grade',colnames(dataNN)))
        idx_ts<-(((i-1)*size_CV+1):(i*size_CV))  ### idx_ts represents the indices of the test set the i-th fold
        X_ts<-dataNN[idx_ts,-targets]  
        Y_ts<-dataNN[idx_ts,targets]      
        
        idx_tr<-setdiff(1:N,idx_ts) ### idx_tr represents  indices of the training sefor the i-th fold
        X_tr<-dataNN[idx_tr,-targets]
        Y_tr<-dataNN[idx_tr,targets]                          
        
        # Computing the correlation between input variables and output variable on the training set
        correlation<-abs(cor(X_tr,Y_tr))
        
        # Initialization : No variables are selected and all the variables are candidates
        selected<-c()
        candidates<-1:n
        
        #mRMR ranks the variables by taking account not only the correlation with the output, but also by avoiding redudant variables
        for (j in 1:n) {
            redundancy_score<-numeric(length(candidates))
            
            if (length(selected)>0) {
                # Compute the correlation between the selected variables and the candidates on the training set
                cor_selected_candidates<-cor(X_tr[,selected,drop=F],X_tr[,candidates,drop=F])
                # Compute the mean correlation for each candidate variable, across the selected variables
                redundancy_score<-apply(cor_selected_candidates,2,mean)
            }
            
            # mRMR: minimum Redundancy Maximum Relevancy
            mRMR_score<-correlation[candidates]-redundancy_score
            
            # Select the candidate variable that maximises the mRMR score
            selected_current<-candidates[which.max(mRMR_score)]
            selected<-c(selected,selected_current)
            
            # Remove the selected variables from the candidates
            candidates<-setdiff(candidates,selected_current)
        }
        
        ranking <- selected

            # Create a dataset including only the first nb_features selected variables
            DS<-cbind(X_tr[,ranking[1:nb_features],drop=F],Y_tr)
            
            # Model fit (using lm function)
            targets<-which(grepl('damage_grade',colnames(DS)))
            normalizer<-layer_normalization(axis = -1L)  %>%  
            adapt(as.matrix(DS))

            neuralmodel <- keras_model_sequential() %>% 
            normalizer  %>% 
            layer_dense(37, activation = 'relu') %>%
            layer_dense(3,activation='softmax')

            neuralmodel %>% compile(
                loss = 'categorical_crossentropy',
                optimizer = optimizer_adam(0.001),
                metrics=c('AUC')
            )
            
            model_history <- neuralmodel %>% fit(
            as.matrix(DS[,-targets]),
            as.matrix(DS[,targets]),
            validation_split = 0.2,
            verbose = 0,
            epochs = 30
            )

            # Model prediction
            yhat <- predict(neuralmodel, as.matrix(X_ts[,ranking[1:nb_features]]))

            yhaty <- classConverter(yhat,Y_ts)

            accuracy<-F1_Score_micro(yhaty[[2]][,],yhaty[[1]][,])
            
            # Cross-validation error = F1
            accuracy_vec[nrow(accuracy_vec)+1,]<-c(nb_features,i,accuracy)
            print(paste("F1-Score Micro -",i,"fold -", nb_features,'features:',accuracy))
            rm('neuralmodel','model_history')
    }

    for (l in seq(2,nb_features,5)){
            meanF1[nrow(meanF1)+1,]<-c(l,mean(filter(accuracy_vec,(n==l))$F1))
    }
    plot(meanF1)
}

accuracy_vec
meanF1

[1] "F1-Score Micro - 1 fold - 2 features: 0.734404536862004"


ERROR: Error in eval(expr, envir, enclos): ValueError: The last dimension of the inputs to a Dense layer should be defined. Found None. Full input shape received: (None, None)



In [8]:
Y_tr

,geo_level_2_mean_damage,geo_level_2_sd_damage,geo_level_3_mean_damage
,<dbl>,<dbl>,<dbl>
1,-0.71177035,-1.75559843,-0.67033928
2,2.06482679,-3.15856423,1.90750849
3,2.08531393,-3.40485914,1.90750849
4,-0.82977394,-0.03013402,-0.28545229
5,-0.05663109,0.08862635,0.55458199
6,-1.61496209,0.74577781,-1.41437293
7,1.27613792,0.04987942,0.96766816
8,-0.32741660,-0.99382566,0.09264440
9,-0.29608325,-1.22466166,-0.40594464
